In [1]:
import numpy as np
import dpluspy 
import pandas

In [2]:
# load data about chromosome lengths and centromere coordinates
bands = pandas.read_csv("cytoBand.txt", sep=r"\s+", names=["chrom", "start", "end", "band", "stain"])
seq_lengths = pandas.read_csv("../hg19.genome", sep=r"\s+", names=["chrom", "seq_length"])

In [3]:
def find_breakpoints(regions, target_size):
    """
    Find points that divide mask `regions` into chunks of approximately
    `target_size` sites each.
    """
    tally = 0
    last_end = 0
    breakpoints = []
    for start, end in regions:
        length = end - start 
        if tally > target_size:
            breakpoints.append(last_end)
            tally = length 
        else: 
            tally += length 
        last_end = end
    if tally >= target_size / 2:
        breakpoints.append(last_end)
    return breakpoints
    

def set_up_intervals(regions, sites_per_window, cen=None):
    """
    
    :param int cen: Position corresponding to the center of the centromere
        (default None)
    """
    if cen is not None: 
        regions0 = regions[regions[:, 1] < cen]
        regions1 = regions[regions[:, 1] >= cen]

        intervals = []

        breakpoints0 = find_breakpoints(regions0, sites_per_window)
        # Account for 1-indexing
        breakpoints0 = np.array([0] + breakpoints0) + 1
        for ii in range(len(breakpoints0) - 1):
            interval = np.array(
                [breakpoints0[ii], breakpoints0[ii + 1], breakpoints0[-1]])
            intervals.append(interval)

        breakpoints1 = find_breakpoints(regions1, sites_per_window)
        # Account for 1-indexing
        breakpoints1 = np.array([cen] + breakpoints1) + 1
        for ii in range(len(breakpoints1) - 1):
            interval = np.array(
                [breakpoints1[ii], breakpoints1[ii + 1], breakpoints1[-1]])
            intervals.append(interval)

    else:
        breakpoints = find_breakpoints(regions, sites_per_window)
        # Account for 1-indexing
        breakpoints = np.array([0] + breakpoints) + 1
        intervals = []
        for ii in range(len(breakpoints) - 1):
            interval = np.array(
                [breakpoints[ii], breakpoints[ii + 1], breakpoints[-1]])
            intervals.append(interval)
    return intervals

In [4]:
# windows tailored to the 1e-4 Morgan exon buffer mask
target_size = 1300000
for ii in range(1, 23):
    mask_fname = f"../bed_files/1e-4M_buffer/mask_chr{ii}.bed.gz"
    regions, _ = dpluspy.utils._read_bed_file(mask_fname)
    if ii in (13, 14, 15, 21, 22):
        intervals = set_up_intervals(regions, target_size)
    else:        
        centromere = bands[(bands["chrom"] == f"chr{ii}") 
                           & (bands["stain"] == "acen")]
        cen = (np.min(centromere["start"]) + np.max(centromere["end"])) / 2
        intervals = set_up_intervals(regions, target_size, cen=cen)
    np.savetxt(f"1e-4M_buffer/windows_chr{ii}.txt", intervals)
    print(f"wrote windows file for chromosome {ii}")

wrote windows file for chromosome 1
wrote windows file for chromosome 2
wrote windows file for chromosome 3
wrote windows file for chromosome 4
wrote windows file for chromosome 5
wrote windows file for chromosome 6
wrote windows file for chromosome 7
wrote windows file for chromosome 8
wrote windows file for chromosome 9
wrote windows file for chromosome 10
wrote windows file for chromosome 11
wrote windows file for chromosome 12
wrote windows file for chromosome 13
wrote windows file for chromosome 14
wrote windows file for chromosome 15
wrote windows file for chromosome 16
wrote windows file for chromosome 17
wrote windows file for chromosome 18
wrote windows file for chromosome 19
wrote windows file for chromosome 20
wrote windows file for chromosome 21
wrote windows file for chromosome 22
